# 03 - Model Experiments

**Project:** {{ cookiecutter.project_name }}

**Objective:** Iterate on models, hyperparameters, and features to beat the baseline.

## Checklist

- [ ] Load preprocessed data
- [ ] Experiment with model architectures and hyperparameters
- [ ] Track experiments with MLflow
- [ ] Compare against baseline
- [ ] Select best model for deployment

In [ ]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

In [ ]:
import json

from pipelines.config.training import TrainingConfig
from pipelines.config.reproducibility import set_all_seeds
from pipelines.evaluation.evaluate import compute_metrics

# Load data
data_dir = project_root / "data" / "processed"
with (data_dir / "train.json").open() as f:
    train_data = json.load(f)
with (data_dir / "val.json").open() as f:
    val_data = json.load(f)
with (data_dir / "test.json").open() as f:
    test_data = json.load(f)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

## 1. Experiment Setup

In [ ]:
# Define experiment configurations to try
experiments = [
    {
        "name": "experiment_v1",
        "description": "First iteration",
        "config": TrainingConfig(
            epochs=5,
            batch_size=16,
            learning_rate=2e-5,
            seed=42,
        ),
    },
    # Add more experiment configs here
]

print(f"Running {len(experiments)} experiment(s)")

## 2. Run Experiments

In [ ]:
results = []

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Running: {exp['name']}")
    print(f"{'='*60}")

    set_all_seeds(exp["config"].seed)

    # TODO: Replace with your model training logic
    # Example:
    # model = YourModel()
    # train_metrics = model.train(train_data, val_data, exp["config"])
    # eval_metrics = model.evaluate(test_data)

    eval_metrics = {"{{ cookiecutter.primary_metric }}": 0.0}  # placeholder
    results.append({"name": exp["name"], **eval_metrics})
    print(f"  Metrics: {eval_metrics}")

## 3. Compare Results

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("{{ cookiecutter.primary_metric }}", ascending=False)
print("\nExperiment Results (sorted by {{ cookiecutter.primary_metric }}):")
results_df

## 4. Threshold Analysis

In [ ]:
{% if cookiecutter.ml_task != 'regression' %}
from pipelines.evaluation.threshold_analysis import sweep_thresholds

# TODO: Replace with actual model scores and labels
# scores = [model.predict(sample["text"]) for sample in test_data]
# labels = [sample["label"] for sample in test_data]
# threshold_results = sweep_thresholds(scores, labels)

# threshold_df = pd.DataFrame(threshold_results)
# threshold_df.plot(x="threshold", y=["precision", "recall", "f1"])
print("Threshold analysis: uncomment code above after training a model")
{% else %}
print("Threshold analysis not applicable for regression tasks")
{% endif %}

## 5. MLflow Comparison

Run experiments via CLI for full MLflow tracking:

```bash
make experiment config=configs/experiment/baseline.yaml
make experiment-all
make compare
make mlflow-ui  # View at http://localhost:5000
```

## 6. Conclusions

| Model | {{ cookiecutter.primary_metric }} | Target (>= {{ cookiecutter.target_metric_value }}) | Status |
|-------|------|--------|--------|
| Baseline | ❓ | {{ cookiecutter.target_metric_value }} | ❓ |
| Experiment v1 | ❓ | {{ cookiecutter.target_metric_value }} | ❓ |

## 7. Next Steps

- [ ] Select best model and register in MLflow Model Registry
- [ ] Update `configs/experiment/` with best configuration
- [ ] Update `pipelines/run_experiment.py` MODEL_REGISTRY with new model adapter
- [ ] Deploy via `docker compose up -d`